# **DATA COLECTION**

## **1. Import Libraries**

In [1]:
import os
import sys
import json
import time
import subprocess
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

## **2. Identify the data source**

### Nguồn 1: Tiki API
- Endpoint danh sách: https://tiki.vn/api/v2/products
- Endpoint chi tiết: https://tiki.vn/api/v2/products/{product_id}
- Cơ chế hiện có trong script: thu thập ID theo category + keyword, sau đó gọi API chi tiết, lưu checkpoint để resume

### Nguồn 2: eBay Browse API
- OAuth token endpoint: https://api.ebay.com/identity/v1/oauth2/token
- Browse search endpoint: https://api.ebay.com/buy/browse/v1/item_summary/search
- Cơ chế hiện có trong script: crawl theo nhiều query, ghép kết quả và loại trùng theo item_id

### Lưu ý vận hành
- Tôn trọng rate limit, thêm delay giữa các request
- eBay yêu cầu APP_ID/CERT_ID hợp lệ
- Tiki crawler có checkpoint giúp chạy nhiều lần để đạt target lớn

In [ ]:
# Standard project paths
PROJECT_ROOT = Path("..").resolve()
CRAWLER_DIR = PROJECT_ROOT / "src" / "crawlers"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

TIKI_SCRIPT = CRAWLER_DIR / "tiki_crawler.py"
EBAY_SCRIPT = CRAWLER_DIR / "ebay_crawler.py"

# Crawl run configuration
CRAWL_CONFIG = {
    "run_tiki": False,          # Set to True to run the Tiki crawler
    "run_ebay": False,          # Set to True to run the eBay crawler
    "tiki_target": 10000,       # Target number of rows for Tiki
    "tiki_max_pages": 120,      # Maximum pages per source (category/keyword)
    "run_timeout": 60 * 60 * 6  # Timeout for each crawl process (seconds)
}

# eBay credentials: prefer environment variables to avoid hard-coding
EBAY_APP_ID = os.getenv("EBAY_APP_ID")
EBAY_CERT_ID = os.getenv("EBAY_CERT_ID")

print("Project root:", PROJECT_ROOT)
print("Crawler directory:", CRAWLER_DIR)
print("Raw data directory:", DATA_RAW_DIR)
print("Tiki script exists:", TIKI_SCRIPT.exists())
print("eBay script exists:", EBAY_SCRIPT.exists())

Project root: C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1
Crawler directory: C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1\src\crawlers
Raw data directory: C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1\data\raw\test
Tiki script exists: True
eBay script exists: True


## **3. Design data structures according to each eBay/Tiki source**

In [3]:
# Source-specific schemas (derived from the two existing crawlers)
tiki_columns = [
    "id", "name", "price", "original_price", "discount_rate", "rating_average",
    "review_count", "quantity_sold", "brand", "category", "url_key"
]

ebay_columns = [
    "query", "item_id", "legacy_item_id", "item_web_url", "item_href",
    "title", "subtitle", "price", "currency", "shipping_cost", "shipping_currency",
    "has_free_shipping", "total_cost", "condition", "condition_id", "buying_options",
    "top_rated_buying_experience", "priority_listing", "adult_only",
    "item_creation_date", "item_end_date", "seller_username", "seller_feedback_score",
    "seller_feedback_percent", "category_id", "category_path", "leaf_category_ids",
    "item_location_country", "item_location_postal", "image_url", "thumbnail_url"
]

script_info = {
    "tiki_crawler": {
        "entry_file": str(TIKI_SCRIPT),
        "list_endpoint": "https://tiki.vn/api/v2/products",
        "detail_endpoint": "https://tiki.vn/api/v2/products/{product_id}",
        "checkpoint_file": "tiki_checkpoint.json",
        "backup_file": "backup.csv",
        "default_output": "products.csv",
        "arguments": ["--target", "--max-pages"],
    },
    "ebay_crawler": {
        "entry_file": str(EBAY_SCRIPT),
        "oauth_endpoint": "https://api.ebay.com/identity/v1/oauth2/token",
        "search_endpoint": "https://api.ebay.com/buy/browse/v1/item_summary/search",
        "default_output": "ebay_products.csv",
        "credentials_required": ["APP_ID", "CERT_ID"],
        "collection_strategy": "multi-query + deduplicate by item_id",
    },
}

print("Tiki schema (", len(tiki_columns), "columns):")
print(tiki_columns)
print("\neBay schema (", len(ebay_columns), "columns):")
print(ebay_columns)
print("\nDetailed information by crawler script:")
print(json.dumps(script_info, indent=2, ensure_ascii=False))

Tiki schema ( 11 columns):
['id', 'name', 'price', 'original_price', 'discount_rate', 'rating_average', 'review_count', 'quantity_sold', 'brand', 'category', 'url_key']

eBay schema ( 31 columns):
['query', 'item_id', 'legacy_item_id', 'item_web_url', 'item_href', 'title', 'subtitle', 'price', 'currency', 'shipping_cost', 'shipping_currency', 'has_free_shipping', 'total_cost', 'condition', 'condition_id', 'buying_options', 'top_rated_buying_experience', 'priority_listing', 'adult_only', 'item_creation_date', 'item_end_date', 'seller_username', 'seller_feedback_score', 'seller_feedback_percent', 'category_id', 'category_path', 'leaf_category_ids', 'item_location_country', 'item_location_postal', 'image_url', 'thumbnail_url']

Detailed information by crawler script:
{
  "tiki_crawler": {
    "entry_file": "C:\\Users\\admin\\OneDrive\\Desktop\\visualize\\DV_Lab1\\src\\crawlers\\tiki_crawler.py",
    "list_endpoint": "https://tiki.vn/api/v2/products",
    "detail_endpoint": "https://tiki.v

## **4. Data collection function**

**Lưu ý:**
- Cập nhật API của eBay APP_ID/CERT_ID trước khi chạy.
- Tiki script hỗ trợ resume bằng checkpoint (tiki_checkpoint.json).
- Nên bật từng nguồn một để dễ theo dõi log.

In [4]:
def run_command(command, cwd, timeout_seconds):
    """Run a shell command and stream the captured output."""
    print("Command:", " ".join(command))
    print("Working directory:", cwd)

    start = time.time()
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        text=True,
        capture_output=True,
        timeout=timeout_seconds,
        check=False,
    )
    elapsed = time.time() - start

    print("=" * 80)
    print("STDOUT:")
    print(completed.stdout if completed.stdout else "<empty>")
    print("=" * 80)
    print("STDERR:")
    print(completed.stderr if completed.stderr else "<empty>")
    print("=" * 80)
    print(f"Exit code: {completed.returncode} | Elapsed: {elapsed:.2f}s")

    return completed.returncode


def run_tiki_crawler(target=10000, max_pages=120, timeout_seconds=7200):
    if not TIKI_SCRIPT.exists():
        raise FileNotFoundError(f"File not found: {TIKI_SCRIPT}")

    command = [
        sys.executable,
        str(TIKI_SCRIPT),
        "--target", str(target),
        "--max-pages", str(max_pages),
    ]
    return run_command(command, cwd=PROJECT_ROOT, timeout_seconds=timeout_seconds)


def run_ebay_crawler(timeout_seconds=7200):
    if not EBAY_SCRIPT.exists():
        raise FileNotFoundError(f"File not found: {EBAY_SCRIPT}")

    # Warn when credentials are not configured yet
    if not EBAY_APP_ID or not EBAY_CERT_ID:
        print("[WARN] EBAY_APP_ID / EBAY_CERT_ID were not found in environment variables.")
        print("[WARN] Update APP_ID/CERT_ID in ebay_crawler.py or export env vars before running.")

    command = [sys.executable, str(EBAY_SCRIPT)]
    return run_command(command, cwd=PROJECT_ROOT, timeout_seconds=timeout_seconds)

## **5. Implement data collection**

Bật/tắt từng nguồn bằng CRAWL_CONFIG ở cell cấu hình. Khi chạy thật, nên chạy theo thứ tự Tiki -> eBay để dễ kiểm tra output.

In [5]:
run_log = {
    "started_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "tiki_exit_code": None,
    "ebay_exit_code": None,
}

if CRAWL_CONFIG["run_tiki"]:
    print("\n" + "=" * 100)
    print("STARTING TIKI CRAWL")
    print("=" * 100)
    run_log["tiki_exit_code"] = run_tiki_crawler(
        target=CRAWL_CONFIG["tiki_target"],
        max_pages=CRAWL_CONFIG["tiki_max_pages"],
        timeout_seconds=CRAWL_CONFIG["run_timeout"],
    )
else:
    print("[INFO] Skipping Tiki crawl (run_tiki=False)")

if CRAWL_CONFIG["run_ebay"]:
    print("\n" + "=" * 100)
    print("STARTING EBAY CRAWL")
    print("=" * 100)
    run_log["ebay_exit_code"] = run_ebay_crawler(
        timeout_seconds=CRAWL_CONFIG["run_timeout"],
    )
else:
    print("[INFO] Skipping eBay crawl (run_ebay=False)")

run_log["finished_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("\nRun log:")
print(json.dumps(run_log, indent=2, ensure_ascii=False))


STARTING TIKI CRAWL
Command: c:\Users\admin\anaconda3\python.exe C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1\src\crawlers\tiki_crawler.py --target 10000 --max-pages 120
Working directory: C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1
STDOUT:
Tổng ID độc nhất hiện có: 14036

--- BƯỚC 2: Tải dữ liệu chi tiết ---
Tiến độ: 3400/10000 sản phẩm, lỗi: 0
Tiến độ: 3500/10000 sản phẩm, lỗi: 0
Tiến độ: 3600/10000 sản phẩm, lỗi: 0
Tiến độ: 3700/10000 sản phẩm, lỗi: 0
Tiến độ: 3800/10000 sản phẩm, lỗi: 0
Tiến độ: 3900/10000 sản phẩm, lỗi: 0
Tiến độ: 4000/10000 sản phẩm, lỗi: 0
Tiến độ: 4100/10000 sản phẩm, lỗi: 0
Tiến độ: 4200/10000 sản phẩm, lỗi: 0
Tiến độ: 4300/10000 sản phẩm, lỗi: 0
Tiến độ: 4400/10000 sản phẩm, lỗi: 0
Tiến độ: 4500/10000 sản phẩm, lỗi: 0
Tiến độ: 4600/10000 sản phẩm, lỗi: 0
Tiến độ: 4700/10000 sản phẩm, lỗi: 0
Tiến độ: 4800/10000 sản phẩm, lỗi: 0
Tiến độ: 4900/10000 sản phẩm, lỗi: 0
Tiến độ: 5000/10000 sản phẩm, lỗi: 0
Tiến độ: 5100/10000 sản phẩm, lỗi: 0
Tiến độ: 52

## **6. Convert to DataFrame**

In [6]:
def read_csv_with_fallback(paths, source_name):
    for path in paths:
        if path.exists():
            df = pd.read_csv(path)
            print(f"[{source_name}] Loaded {len(df):,} rows from: {path}")
            return df, path
    print(f"[{source_name}] Data file not found.")
    return pd.DataFrame(), None

tiki_candidates = [
    DATA_RAW_DIR / "products.csv",
    DATA_RAW_DIR / "tiki_products.csv",
    PROJECT_ROOT / "products.csv",
    PROJECT_ROOT / "backup.csv",
]

ebay_candidates = [
    DATA_RAW_DIR / "ebay_products.csv",
    PROJECT_ROOT / "ebay_products.csv",
]

df_tiki, tiki_source_path = read_csv_with_fallback(tiki_candidates, "Tiki")
df_ebay, ebay_source_path = read_csv_with_fallback(ebay_candidates, "eBay")

print("\nDataset shapes:")
print("- Tiki:", df_tiki.shape)
print("- eBay:", df_ebay.shape)

[Tiki] Loaded 6,700 rows from: C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1\products.csv
[eBay] Data file not found.

Dataset shapes:
- Tiki: (6700, 11)
- eBay: (0, 0)


## **7. Initial data quality check**

Kiểm tra gồm: cấu trúc cột, thống kê số học, missing values, và trùng lặp theo khóa chính của từng nguồn.

In [7]:
def profile_dataframe(df, name, max_numeric_cols=12):
    print("\n" + "=" * 100)
    print(f"[{name}] OVERVIEW")
    print("=" * 100)
    print("Shape:", df.shape)

    if df.empty:
        print("DataFrame is empty, skipping profiling.")
        return

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nInfo:")
    df.info()

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        use_cols = numeric_cols[:max_numeric_cols]
        print(f"\nNumeric describe ({len(use_cols)}/{len(numeric_cols)} columns):")
        display(df[use_cols].describe().T)
    else:
        print("\nNo numeric columns available for describe().")

profile_dataframe(df_tiki, "Tiki")
profile_dataframe(df_ebay, "eBay")


[Tiki] OVERVIEW
Shape: (6700, 11)

Columns:
['id', 'name', 'price', 'original_price', 'discount_rate', 'rating_average', 'review_count', 'quantity_sold', 'brand', 'category', 'url_key']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6700 entries, 0 to 6699
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              6700 non-null   int64  
 1   name            6700 non-null   object 
 2   price           6700 non-null   int64  
 3   original_price  6700 non-null   int64  
 4   discount_rate   6700 non-null   int64  
 5   rating_average  6700 non-null   float64
 6   review_count    6700 non-null   int64  
 7   quantity_sold   6700 non-null   int64  
 8   brand           6700 non-null   object 
 9   category        6700 non-null   object 
 10  url_key         6700 non-null   object 
dtypes: float64(1), int64(6), object(4)
memory usage: 575.9+ KB

Numeric describe (7/7 columns):


,count,mean,std,min,25%,50%,75%,max
id,6700.0,1.956456e+08,9.773153e+07,126003.0,1.101253e+08,260515117.5,2.766290e+08,279280455.0
price,6700.0,5.315268e+05,1.488954e+06,1000.0,8.900000e+04,176000.0,3.902500e+05,42250000.0
original_price,6700.0,6.061589e+05,1.649414e+06,1000.0,9.900000e+04,195000.0,4.490000e+05,42250000.0
discount_rate,6700.0,9.268358e+00,1.483583e+01,0.0,0.000000e+00,0.0,1.800000e+01,75.0
rating_average,6700.0,2.875866e+00,2.329335e+00,0.0,0.000000e+00,4.5,5.000000e+00,5.0
review_count,6700.0,4.600940e+01,2.522620e+02,0.0,0.000000e+00,1.0,1.000000e+01,7116.0
quantity_sold,6700.0,8.528546e+02,9.752403e+03,0.0,2.000000e+00,13.0,8.800000e+01,553719.0



[eBay] OVERVIEW
Shape: (0, 0)
DataFrame is empty, skipping profiling.


In [8]:
def missing_report(df, name):
    print("\n" + "-" * 100)
    print(f"[{name}] MISSING VALUES")
    print("-" * 100)

    if df.empty:
        print("DataFrame is empty.")
        return pd.DataFrame(columns=["missing_count", "missing_pct"])

    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2)
    report = pd.DataFrame({
        "missing_count": missing_count,
        "missing_pct": missing_pct
    }).sort_values("missing_count", ascending=False)

    report = report[report["missing_count"] > 0]
    if report.empty:
        print("No missing values found.")
    else:
        display(report)
    return report

missing_tiki = missing_report(df_tiki, "Tiki")
missing_ebay = missing_report(df_ebay, "eBay")


----------------------------------------------------------------------------------------------------
[Tiki] MISSING VALUES
----------------------------------------------------------------------------------------------------
No missing values found.

----------------------------------------------------------------------------------------------------
[eBay] MISSING VALUES
----------------------------------------------------------------------------------------------------
DataFrame is empty.


In [9]:
def duplicate_report(df, name, key_col):
    print("\n" + "-" * 100)
    print(f"[{name}] DUPLICATE CHECK by {key_col}")
    print("-" * 100)

    if df.empty:
        print("DataFrame is empty.")
        return df

    if key_col not in df.columns:
        print(f"Key column not found: {key_col}")
        return df

    dup_count = df.duplicated(subset=[key_col]).sum()
    print(f"Number of duplicate key rows: {dup_count}")

    if dup_count > 0:
        print("Removing duplicates and keeping the first occurrence.")
        dedup_df = df.drop_duplicates(subset=[key_col], keep="first")
        print("Shape before/after:", df.shape, "->", dedup_df.shape)
        return dedup_df

    return df

df_tiki = duplicate_report(df_tiki, "Tiki", "id")
df_ebay = duplicate_report(df_ebay, "eBay", "item_id")


----------------------------------------------------------------------------------------------------
[Tiki] DUPLICATE CHECK by id
----------------------------------------------------------------------------------------------------
Number of duplicate key rows: 0

----------------------------------------------------------------------------------------------------
[eBay] DUPLICATE CHECK by item_id
----------------------------------------------------------------------------------------------------
DataFrame is empty.


## **8. Saved Raw Data**

Lưu dữ liệu theo từng nguồn:
- data/raw/tiki_products.csv
- data/raw/ebay_products.csv

In [10]:
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

tiki_output = DATA_RAW_DIR / "tiki_products.csv"
ebay_output = DATA_RAW_DIR / "ebay_products.csv"

save_report = {
    "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "tiki": {
        "source_file": str(tiki_source_path) if tiki_source_path else None,
        "output_file": str(tiki_output),
        "rows": 0,
        "columns": 0,
    },
    "ebay": {
        "source_file": str(ebay_source_path) if ebay_source_path else None,
        "output_file": str(ebay_output),
        "rows": 0,
        "columns": 0,
    },
}

if not df_tiki.empty:
    df_tiki.to_csv(tiki_output, index=False, encoding="utf-8-sig")
    save_report["tiki"]["rows"] = int(df_tiki.shape[0])
    save_report["tiki"]["columns"] = int(df_tiki.shape[1])
    print(f"Saved Tiki raw data: {tiki_output} ({len(df_tiki):,} rows)")
else:
    print("[INFO] No Tiki data available to save.")

if not df_ebay.empty:
    df_ebay.to_csv(ebay_output, index=False, encoding="utf-8-sig")
    save_report["ebay"]["rows"] = int(df_ebay.shape[0])
    save_report["ebay"]["columns"] = int(df_ebay.shape[1])
    print(f"Saved eBay raw data: {ebay_output} ({len(df_ebay):,} rows)")
else:
    print("[INFO] No eBay data available to save.")

print("\nData saving report:")
print(json.dumps(save_report, indent=2, ensure_ascii=False))

if not df_tiki.empty:
    print("\n[Tiki] Preview:")
    display(df_tiki.head())

if not df_ebay.empty:
    print("\n[eBay] Preview:")
    display(df_ebay.head())

Saved Tiki raw data: C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1\data\raw\test\tiki_products.csv (6,700 rows)
[INFO] No eBay data available to save.

Data saving report:
{
  "saved_at": "2026-04-13 20:11:47",
  "tiki": {
    "source_file": "C:\\Users\\admin\\OneDrive\\Desktop\\visualize\\DV_Lab1\\products.csv",
    "output_file": "C:\\Users\\admin\\OneDrive\\Desktop\\visualize\\DV_Lab1\\data\\raw\\test\\tiki_products.csv",
    "rows": 6700,
    "columns": 11
  },
  "ebay": {
    "source_file": null,
    "output_file": "C:\\Users\\admin\\OneDrive\\Desktop\\visualize\\DV_Lab1\\data\\raw\\test\\ebay_products.csv",
    "rows": 0,
    "columns": 0
  }
}

[Tiki] Preview:


,id,name,price,original_price,discount_rate,rating_average,review_count,quantity_sold,brand,category,url_key
0,143116837,Dây Lẻ Đàn Guitar Acoustic 1-2-3-4-5-6,8500,8500,0,4.6,28,531,Alice,Nhà Cửa - Đời Sống,day-le-dan-guitar-acoustic-1-2-3-4-5-6-p143116837
1,277165432,"Sách Hành Lang Hẹp - Nhà Nước, Xã Hội Và Vận Mệnh Của Tự Do",274000,397000,31,5.0,37,567,No Brand,"Sách quản trị, lãnh đạo",sach-hanh-lang-hep-nha-nuoc-xa-hoi-va-van-menh-cua-tu-do-p277165432
2,276609755,Valen Gile - Áo gile form ôm thiết kế cao cấp WHITE CHIC,650000,650000,0,0.0,0,0,WHITE CHIC,Thời trang nữ,valen-gile-ao-gile-form-om-thiet-ke-cao-cap-white-chic-p276609755
3,277962612,Sách Thiên Quan Tứ Phúc,112800,179000,37,5.0,43,314,No Brand,Truyện Chữ Đam Mỹ,sach-thien-quan-tu-phuc-p277962612
4,277445975,Giá Treo Khăn Nhà Tắm Dán Tường 2 Thanh INOX Giá Treo Vung Nồi Móc Đồ Nhà Bếp Tiện Lợi,35000,69000,49,0.0,0,5,GUANG MING,Nhà Cửa - Đời Sống,gia-treo-khan-nha-tam-dan-tuong-2-thanh-inox-gia-treo-vung-noi-moc-do-nha-bep-tien-loi-p277445975


## **9. Summary**

Notebook 01 đã hoàn thiện theo luồng làm việc:
1. Cấu hình nguồn và tham số thu thập
2. Gọi script crawl tương ứng cho từng nguồn
3. Đọc dữ liệu thô riêng theo nguồn
4. Kiểm tra chất lượng ban đầu (missing, duplicate, profile) riêng theo nguồn
5. Lưu output riêng vào data/raw

**Output chính:**
- data/raw/tiki_products.csv
- data/raw/ebay_products.csv

**Ghi chú theo từng script:**
- Tiki: có checkpoint + backup, phù hợp chạy nhiều lần để đạt target lớn.
- eBay: cần credential hợp lệ, thu thập theo nhiều query và loại trùng theo item_id.

**Bước tiếp theo:**
- Notebook 02: tiền xử lý riêng cho từng nguồn.